# Colab Virtual VRAM 测试
直接在 Colab 上测试 APT-Transformer Virtual VRAM

In [ ]:
# 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. 设置 root 密码
!echo root:colab123 | chpasswd
!mkdir -p /var/run/sshd
!echo 'PermitRootLogin yes' >> /etc/ssh/sshd_config
!echo 'PasswordAuthentication yes' >> /etc/ssh/sshd_config

In [ ]:
# 3. 启动 SSH 服务器
!service ssh start
!echo 'SSH 服务器已启动'

In [ ]:
# 4. 下载并启动 cloudflared
import subprocess
import time

# 下载 cloudflared 二进制
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

# 启动 cloudflared 隧道（后台运行）
import os
os.environ['TUNNEL_URL'] = ''

# 使用 nohup 在后台启动
!nohup cloudflared tunnel --url tcp://localhost:22 > /tmp/tunnel.log 2>&1 &

print('等待 10 秒让隧道建立...')
time.sleep(10)

# 读取日志获取公网地址
!cat /tmp/tunnel.log

In [ ]:
# 5. 获取 SSH 连接信息
print("=" * 60)
print("SSH 隧道信息：")
print("=" * 60)

!cat /tmp/tunnel.log | grep -E 'trycloudflare|\.com|https?://'

print("\n" + "=" * 60)
print("使用方法：")
print("=" * 60)
print("1. 复制上面的 .trycloudflare.com 地址")
print("2. 在本地终端运行:")
print("   ssh -p 22 root@<trycloudflare地址>")
print("   密码: colab123")
print("=" * 60)

In [ ]:
# 6. 克隆你的代码仓库
# 注意：需要先推送到 GitHub 或从 Google Drive 复制

# 方式 1：从 GitHub 克隆
# !git clone https://github.com/your-username/APT-Transformer.git

# 方式 2：从 Google Drive 复制
from google.colab import drive
drive.mount('/content/drive')

# 复制代码（假设你已上传到 Drive）
!cp -r /content/drive/MyDrive/APT-Transformer /content/ 2>/dev/null || echo '请先上传代码到 Google Drive'

%cd /content/APT-Transformer

In [ ]:
# 7. 安装依赖
!pip install torch datasets transformers accelerate -q
!pip install huggingface_hub -q

In [ ]:
# 8. 检查 GPU
!nvidia-smi

In [ ]:
# 9. 运行测试（等待 SSH 连接后）
# 或直接在这里运行

!python -m apt.trainops.scripts.pretrain_quickcook \
    --output-dir ./test_vvram_lecac \
    --max-steps 50 \
    --batch-size 2 \
    --gradient-accumulation 2 \
    --use-virtual-vram \
    --vram-enable-nested-v16 \
    --vram-verbose